# Lesson 30 Lab — From Slow Subgraph to Deliverable Kernel

**Puzzle:** When baseline, fusion, validation, performance gate, and rollback change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates baseline, fusion, validation, performance gate, and rollback and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

A deliverable kernel is more than code that runs. It has a frozen baseline, explicit input contract, adversarial correctness suite, latency distribution, environment identity, stop condition, integration path, and rollback. Optimization ends when the declared gate is met.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["baseline, fusion, validation, performance gate, and rollback"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: reviewed Triton kernel or explicit model described below.

Without a stop condition, kernel work can continue indefinitely while integration risk and maintenance cost grow.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 30
LESSON_TITLE = 'From Slow Subgraph to Deliverable Kernel'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260843
}


## 5. Freeze the experiment

**Experiment:** Deliver a tanh-approximate GELU Triton kernel with correctness tolerance, repeated timings, minimum performance gate, and PyTorch rollback.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 0.5542071330004571,
  "secondary": 0.0197759997099638,
  "max_abs_error": 4.76837158203125e-07,
  "passed": false,
  "details": {
    "pytorch_median_ms": 0.010960000101476908,
    "correctness_tolerance": 0.0001,
    "minimum_speed_ratio": 0.8,
    "rollback": "torch.nn.functional.gelu",
    "triton_samples_ms": [
      0.02969600073993206,
      0.02316799946129322,
      0.02179200015962124,
      0.019648000597953796,
      0.018624000251293182,
      0.023231999948620796,
      0.0197759997099638,
      0.019360000267624855,
      0.018912000581622124,
      0.020416000857949257,
      0.020287999883294106,
      0.01820800080895424,
      0.020640000700950623,
      0.01833599992096424,
      0.01865600049495697,
      0.020191999152302742,
      0.0197759997099638,
      0.019392000511288643,
      0.019392000511288643,
      0.01942400075495243,
      0.019328000023961067,
      0.020640000700950623,
      0.020447999238967896,
      0.019519999623298645,
      0

## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| PyTorch/Triton speed ratio | 0.554x |
| Triton median | 0.0198 ms |
| Maximum absolute error | 4.768e-07 |
| Acceptance gate | false |


## 8. Explain without overclaiming

The deliverable GELU reached 0.55x of the PyTorch baseline with 4.77e-07 max error, so its frozen correctness and no-catastrophic-regression gate was False.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Ship only when correctness and performance gates pass together; otherwise keep the documented PyTorch fallback.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 30,
  "title": "From Slow Subgraph to Deliverable Kernel",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260843
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 0.5542071330004571,
    "secondary": 0.0197759997099638,
    "max_abs_error": 4.76837158203125e-07,
    "passed": false,
    "details": {
      "pytorch_median_ms": 0.010960000101476908,
      "correctness_tolerance": 0.0001,
      "minimum_speed_ratio": 0.8,
      "rollback": "torch.nn.functional.gelu",
      "triton_samples_ms": [
        0.02969600073993206,
        0.02316799946129322,
        0.02179200015962124,
        0.019648000597953796,
        0.018624000251293182,
        0.023231999948620796,
        0.0197759997099638,
        0.01936000026762

## 10. Make the bounded decision

> Ship only when correctness and performance gates pass together; otherwise keep the documented PyTorch fallback.

**Failure analysis:** Without a stop condition, kernel work can continue indefinitely while integration risk and maintenance cost grow.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
